# Lesson 5: Human in the Loop

Note: This notebook is running in a later version of langgraph that it was filmed with. The later version has a couple of key additions:
- Additional state information is stored to memory and displayed when using `get_state()` or `get_state_history()`.
- State is additionally stored every state transition while previously it was stored at an interrupt or at the end.
These change the command output slightly, but are a useful addtion to the information available.

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.sqlite import SqliteSaver

import sqlite3
db_path = 'checkpoints.db'
conn = sqlite3.connect(db_path, check_same_thread=False)
memory = SqliteSaver(conn)

#memory = SqliteSaver.from_conn_string(":memory:")
#memory = SqliteSaver.from_conn_string('checkpoints.db')


In [2]:
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage

"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # merge the new messages with the existing messages
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # append any new messages to the end
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [3]:
#tool = TavilySearchResults(max_results=2)
from langchain_community.tools import DuckDuckGoSearchRun
tool = DuckDuckGoSearchRun()

## Manual human approval

In [4]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [5]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
base_url = "http://127.0.0.1:1234/v1" 
model_name ="google/gemma-3-4b"
model = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)
#model = ChatOpenAI(model="gpt-3.5-turbo")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [5]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

NameError: name 'abot' is not defined

In [7]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='05774f27-11ff-439f-8679-444d99c48cba'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '469428286', 'function': {'arguments': '{"query":"weather in San Francisco"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-z9t3b19p7gfnlzh21s8eme', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--8733be15-de26-4850-b6ac-8df7c8ddda69-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '469428286', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 

In [8]:
abot.graph.get_state(thread).next

('action',)

### continue after interrupt

In [9]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '838311847', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content="8 Jul 2025 — San Francisco, CA Forecast · Morning. 57°. -- · Afternoon. 61°. -- · Evening. 57°. Chance of Rain3% · Overnight. 54°. Chance of Rain12%. 4 hours ago — First Alert Weather · Current Conditions. San Francisco, CA . °F°C. Partly Cloudy. Partly Cloudy61°. 60°. Feels Like. W 9mph. Wind. 93%. Humidity. 9mi. 15 Jun 2025 — Get the monthly weather forecast for San Francisco , CA, including daily high/low, historical averages, to help you plan ahead. 1 day ago — Weather for the next 24 hours ; clear sky with few clouds. 14 °C · 0 mm 0 %. W · 5 km/h. clear sky with few clouds. 13 °C · 0 mm 0 %. W · 5 km/h. clear sky with few ... 3 days ago — San Francisco is set for one of its hottest stretches of the year , with highs in the 80s and 90s. Here's how long the summerlike heat will last.", name='duckduckgo_sea

In [11]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='05774f27-11ff-439f-8679-444d99c48cba'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '469428286', 'function': {'arguments': '{"query":"weather in San Francisco"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-z9t3b19p7gfnlzh21s8eme', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--8733be15-de26-4850-b6ac-8df7c8ddda69-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '469428286', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 

In [12]:
abot.graph.get_state(thread).next

()

In [31]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='2d7cabd3-700d-4aba-9a35-7c12463af4b4'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '551502797', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-pnv98ru6v14k6fa4xlrjl', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--855f37b0-3d7d-4f1c-9c2c-91993a842a62-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in Los Angeles'}, 'id': '551502797', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 533, 'input_token_details'

proceed? n


aborting


## Modify State
Run until the interrupt and then modify the state.

In [20]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 533,

In [21]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36,

In [22]:
current_values = abot.graph.get_state(thread)

In [23]:
current_values.values['messages'][-1]

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '779444536', 'function': {'arguments': '{"query":"Los Angeles weather today"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 656, 'total_tokens': 692, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-hklkuayiak6o4sswzgy1is', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--b4cdb58a-a01d-4e55-bf7c-fb6239362693-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'Los Angeles weather today'}, 'id': '779444536', 'type': 'tool_call'}], usage_metadata={'input_tokens': 656, 'output_tokens': 36, 'total_tokens': 692, 'input_token_details': {}, 'output_token_details': {}})

In [24]:
current_values.values['messages'][-1].tool_calls

[{'name': 'duckduckgo_search',
  'args': {'query': 'Los Angeles weather today'},
  'id': '779444536',
  'type': 'tool_call'}]

In [25]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'duckduckgo_search',
  'args': {'query': 'current weather in Louisiana'},
  'id': _id}
]

In [26]:
abot.graph.update_state(thread, current_values.values)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 533,

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f09479b-30be-664a-800d-1ac5f036be8d'}}

In [27]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36,

In [28]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'current weather in Louisiana'}, 'id': '779444536', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='6 hours ago — Hourly Weather-Baton Rouge, LA · 1 am. Partly Cloudy. 71°. 3%. S 0 mph. Partly Cloudy. Feels Like71°. WindS 0 mph. Humidity93%. UV Index0 of 11. Cloud Cover34%. 2 days ago — Halifax, NS 18°. New Orleans, LA Current Weather . New Orleans, LA. Updated 8 minutes ago. 26. °C. Clear. Feels 26. H: 32° L: 24°. Hourly . Full 72 hours · 12am. 6 days ago — Hurricane. Potential tropical development in Gulf poses primary U.S. threat · Weather Forecasts. Highs 90-100 F to challenge heat records in Mississippi Valley. 19 Nov 2024 — Weather ; Thursday. Mostly Sunny. 92° · 68°. 3% ; Friday. Mostly Sunny. 93° · 69°. 8% ; Saturday. Mostly Sunny. 93° · 72°. 11%. 2 days ago — Providing a local hourly Louisiana weather forecast of rain, sun, wind, humidity and temperature.', name='duckduckgo_search', id='b5ab459

## Time Travel

In [29]:
states = []
for state in abot.graph.get_state_history(thread):
    print(state)
    print('--')
    states.append(state)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36,

To fetch the same state as was filmed, the offset below is changed to `-3` from `-1`. This accounts for the initial state `__start__` and the first state that are now stored to state memory with the latest version of software.

In [30]:
to_replay = states[-3]

In [32]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in Los Angeles'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 533,

In [33]:
for event in abot.graph.stream(None, to_replay.config):
    for k, v in event.items():
        print(v)

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in Los Angeles'}, 'id': '347692029', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='13 hours ago — Most of Los Angeles and Ventura counties are expected to get a half-inch of rain between Wednesday and Friday, with mountain areas seeing around an inch at most ... 5 days ago — Current weather condition is Clear with real-time temperature (26°C), humidity 58%, wind 4.3km/h, pressure (1011mb), UV (0), visibility (16km) in ... 15 Jun 2025 — Get the monthly weather forecast for Los Angeles, CA , including daily high/low, historical averages, to help you plan ahead. 2 days ago — Cloudy skies with periods of rain late . Low 67F. Winds light and variable. Chance of rain 100%. Rainfall around a quarter of an inch. 5 Apr 2025 — Cloudy with periods of rain . Low 69F. Winds light and variable. Chance of rain 80%. Humidity80%. UV Index0 of 11. Moonrise2 ...', name='duckduckgo_search', id='201e93c2-262a-421

## Go back in time and edit

In [34]:
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in Los Angeles'}, 'id': '347692029', 'type': 'tool_call'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 533,

In [35]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']
to_replay.values['messages'][-1].tool_calls = [{'name': 'duckduckgo_search,
  'args': {'query': 'current weather in LA, accuweather'},
  'id': _id}]

In [36]:
branch_state = abot.graph.update_state(to_replay.config, to_replay.values)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='e787f6f6-f8e2-451b-80e8-32f11fec30f8'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '347692029', 'function': {'arguments': '{"query":"weather in Los Angeles"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 497, 'total_tokens': 533, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-j3zitxbsltcshuy8mizjaa', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2abc3689-7f20-47be-8827-0e7947ce6fd1-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029'}], usage_metadata={'input_tokens': 497, 'output_tokens': 36, 'total_tokens': 533, 'input_token_details

In [37]:
for event in abot.graph.stream(None, branch_state):
    for k, v in event.items():
        if k != "__end__":
            print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current weather in LA, accuweather'}, 'id': '347692029', 'type': 'tool_call'}


KeyError: 'tavily_search_results_json'

## Add message to a state at a given time

In [ ]:
to_replay

In [ ]:
_id = to_replay.values['messages'][-1].tool_calls[0]['id']

In [ ]:
state_update = {"messages": [ToolMessage(
    tool_call_id=_id,
    name='duckduckgo_search',
    content="54 degree celcius",
)]}

In [ ]:
branch_and_add = abot.graph.update_state(
    to_replay.config, 
    state_update, 
    as_node="action")

In [ ]:
for event in abot.graph.stream(None, branch_and_add):
    for k, v in event.items():
        print(v)

# Extra Practice

## Build a small graph
This is a small simple graph you can tinker with if you want more insight into controlling state memory.

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [6]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langgraph.checkpoint.sqlite import SqliteSaver

Define a simple 2 node graph with the following state:
-`lnode`: last node
-`scratch`: a scratchpad location
-`count` : a counter that is incremented each step

In [7]:
class AgentState(TypedDict):
    lnode: str
    scratch: str
    count: Annotated[int, operator.add]

In [8]:
def node1(state: AgentState):
    print(f"node1, count:{state['count']}")
    return {"lnode": "node_1",
            "count": 1,
           }
def node2(state: AgentState):
    print(f"node2, count:{state['count']}")
    return {"lnode": "node_2",
            "count": 1,
           }

The graph goes N1->N2->N1... but breaks after count reaches 3.

In [9]:
def should_continue(state):
    return state["count"] < 3

In [10]:
builder = StateGraph(AgentState)
builder.add_node("Node1", node1)
builder.add_node("Node2", node2)

builder.add_edge("Node1", "Node2")
builder.add_conditional_edges("Node2", 
                              should_continue, 
                              {True: "Node1", False: END})
builder.set_entry_point("Node1")

In [11]:
#memory = SqliteSaver.from_conn_string(":memory:")
memory = SqliteSaver.from_conn_string('checkpoints.db')
graph = builder.compile(checkpointer=memory)

### Run it!
Now, set the thread and run!

In [12]:
thread = {"configurable": {"thread_id": str(1)}}
graph.invoke({"count":0, "scratch":"hi"},thread)

AttributeError: '_GeneratorContextManager' object has no attribute 'get_next_version'

### Look at current state

Get the current state. Note the `values` which are the AgentState. Note the `config` and the `thread_ts`. You will be using those to refer to snapshots below.

In [13]:
graph.get_state(thread)

AttributeError: '_GeneratorContextManager' object has no attribute 'get_tuple'

View all the statesnapshots in memory. You can use the displayed `count` agentstate variable to help track what you see. Notice the most recent snapshots are returned by the iterator first. Also note that there is a handy `step` variable in the metadata that counts the number of steps in the graph execution. This is a bit detailed - but you can also notice that the *parent_config* is the *config* of the previous node. At initial startup, additional states are inserted into memory to create a parent. This is something to check when you branch or *time travel* below.

### Look at state history

In [ ]:
for state in graph.get_state_history(thread):
    print(state, "\n")

Store just the `config` into an list. Note the sequence of counts on the right. `get_state_history` returns the most recent snapshots first.

In [ ]:
states = []
for state in graph.get_state_history(thread):
    states.append(state.config)
    print(state.config, state.values['count'])

Grab an early state.

In [ ]:
states[-3]

This is the state after Node1 completed for the first time. Note `next` is `Node2`and `count` is 1.

In [ ]:
graph.get_state(states[-3])

### Go Back in Time
Use that state in `invoke` to go back in time. Notice it uses states[-3] as *current_state* and continues to node2,

In [ ]:
graph.invoke(None, states[-3])

Notice the new states are now in state history. Notice the counts on the far right.

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state.config, state.values['count'])

You can see the details below. Lots of text, but try to find the node that start the new branch. Notice the parent *config* is not the previous entry in the stack, but is the entry from state[-3].

In [ ]:
thread = {"configurable": {"thread_id": str(1)}}
for state in graph.get_state_history(thread):
    print(state,"\n")

### Modify State
Let's start by starting a fresh thread and running to clean out history.

In [ ]:
thread2 = {"configurable": {"thread_id": str(2)}}
graph.invoke({"count":0, "scratch":"hi"},thread2)

In [ ]:
from IPython.display import Image

Image(graph.get_graph().draw_png())

In [ ]:
states2 = []
for state in graph.get_state_history(thread2):
    states2.append(state.config)
    print(state.config, state.values['count'])   

Start by grabbing a state.

In [ ]:
save_state = graph.get_state(states2[-3])
save_state

Now modify the values. One subtle item to note: Recall when agent state was defined, `count` used `operator.add` to indicate that values are *added* to the current value. Here, `-3` will be added to the current count value rather than replace it.

In [ ]:
save_state.values["count"] = -3
save_state.values["scratch"] = "hello"
save_state

Now update the state. This creates a new entry at the *top*, or *latest* entry in memory. This will become the current state.

In [ ]:
graph.update_state(thread2,save_state.values)

Current state is at the top. You can match the `thread_ts`.
Notice the `parent_config`, `thread_ts` of the new node - it is the previous node.

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

### Try again with `as_node`
When writing using `update_state()`, you want to define to the graph logic which node should be assumed as the writer. What this does is allow th graph logic to find the node on the graph. After writing the values, the `next()` value is computed by travesing the graph using the new state. In this case, the state we have was written by `Node1`. The graph can then compute the next state as being `Node2`. Note that in some graphs, this may involve going through conditional edges!  Let's try this out.

In [ ]:
graph.update_state(thread2,save_state.values, as_node="Node1")

In [ ]:
for i, state in enumerate(graph.get_state_history(thread2)):
    if i >= 3:  #print latest 3
        break
    print(state, '\n')

`invoke` will run from the current state if not given a particular `thread_ts`. This is now the entry that was just added.

In [ ]:
graph.invoke(None,thread2)

Print out the state history, notice the `scratch` value change on the latest entries.

In [ ]:
for state in graph.get_state_history(thread2):
    print(state,"\n")

Continue to experiment!